# <span style="color:darkblue"> Project 1 - Scraping the Healthy Food Policy Project Database </span>

<font size = "3">
 
### DATASCI 530 · Project 1 · Whitney Barr


## Project purpose

I use this notebook to collect information about food policies from the Healthy Food Policy Project database and save it in a table for later review. This project focuses on collecting and organizing the information, not evaluating whether the policies work.

## Approach

I use Selenium to open the database in Chrome and visit each policy page. I collect the same fields from every page so the results can be compared in one table. In plain terms, the code opens a browser, finds the page links, reads selected text, and saves the results.

- Website: https://healthyfoodpolicyproject.org/policy-database (permission obtained for this health-policy website)
- Data collected: jurisdiction name, jurisdiction type, state, policy category, summary text, and policy URL
- The site's ``` robots.txt ``` allows policy pages and asks for a 10-second delay between requests, so the code waits 10 seconds between policy-page requests

The website may change over time, so a later run may collect different information or require changes to the code.

# <span style="color:darkblue"> I. Import Packages </span>

I import the tools this project uses. Selenium controls Chrome, `time` adds the pause requested by the website, and pandas organizes the results into a table and saves them as a CSV file. A CSV is a common table file that can be opened in spreadsheet programs.

In [17]:
# Manage datasets
import pandas as pd

# Work with time data
import time

# Automate navigating within the browser (Selenium)
from selenium import webdriver

# <span style="color:darkblue"> II. Running Chrome from Python </span>

<font size = "5">

Initialize web driver and open the website

<font size = "3">

- This opens a Google Chrome window. Keep it open while the notebook is running.
- I use Selenium to make Chrome open pages and read information from them.
- I start on the database page because it contains links to the individual policy pages.

In [18]:
options = webdriver.ChromeOptions()

driver = webdriver.Chrome(options=options)

starting_url = 'https://healthyfoodpolicyproject.org/policy-database'

driver.get(starting_url)

# <span style="color:darkblue"> III. Find the Policy Detail Pages </span>

<font size = "5">

Find and save the links to the policy pages

<font size = "3">

- Each policy card has a "View Details" link. The line `list_links = driver.find_elements('xpath','//div[@class = "policy-card-button-wrap"]/a')` asks Chrome to find all the links inside the policy-card button areas.
- In the search directions, `//div[@class = "policy-card-button-wrap"]` means "find each policy-card button area," and `/a` means "within it, select the link." XPath is the set of directions Selenium uses to locate those parts of the webpage.
- `.find_elements()` returns the matching links as a list. `.get_attribute("href")` then reads the web address from each link.
- I save the addresses in a list before leaving the database page. That way, the code can open each policy page directly.

I save the web addresses as text so I can use them after Chrome leaves the database page.

In [19]:
list_links = driver.find_elements('xpath','//div[@class = "policy-card-button-wrap"]/a')

# Count how many elements are extracted
num_links = len(list_links)
print(num_links)

# Store the web address of each policy
list_urls = []
for i in range(num_links):
    list_urls.append(list_links[i].get_attribute("href"))

# Display the first policy web addresses
list_urls[0:3]

605


['https://healthyfoodpolicyproject.org/policy/abingdon-va-code-%c2%a7-18-132-current-through-april-21-2016',
 'https://healthyfoodpolicyproject.org/policy/alachua-county-fla-code-of-ordinances-%c2%a7-407-41-current-as-of-may-8-2019',
 'https://healthyfoodpolicyproject.org/policy/alachua-county-fla-third-amendment-to-emergency-order-no-2020-30-%c2%a7%c2%a7-2-3-july-17-2020']

# <span style="color:darkblue"> IV. Test on a Single Policy </span>

<font size = "5">

Check the extraction on one policy page before collecting them all

<font size = "3">

- I open the first policy page using its saved web address.
- Each field has a label, such as **State**, and a value. I use the label to find the right field and read its text.
- The text includes the label, so I remove the label and keep the value. I also clean up extra spaces.
- If a field is not present, the function returns a blank value so the other fields can still be collected.

I wrote two functions so I can reuse the same steps on every policy page. `get_field(label)` finds and cleans one labeled value. `get_summary()` reads the summary paragraph. If either function cannot find its text, it returns a blank value instead of stopping collection of the other fields.

### What happens inside `get_field(label)`

The function works through the field in these steps:

1. `if len(elements) == 0` checks whether the page search found a matching field. If it found none, `return ""` gives back a blank value instead of trying to read something that is not there.
2. `elements[0].text` takes the text from the first matching field. `.replace(label, "", 1)` removes the label one time. For example, `State California` becomes ` California`.
3. `" ".join(value.split())` tidies the text: `.split()` separates it wherever there is a space or line break, and `.join()` puts the pieces back together with one space between them. On the first policy page, the simpler version without this step left the value as `\nVirginia` because the label and value were on separate lines. With this step, the result is `Virginia`. This is better for the dataset because the value has no extra line break before it and is easier to read, compare, and use in a table.
4. The function returns the cleaned value. Any punctuation in the value is left as it appears on the webpage.

The search directions compare the bold label directly with the text I provide, such as `State`. This keeps the XPath simple, but the label must match the webpage text exactly, including its spacing.

I check the results on one policy first. This gives me a chance to see whether the right information is being collected before I repeat the process for every page.

In [20]:
# Extract the value of one labelled field on the policy page
def get_field(label):
    xpath = '//div[contains(@class,"policy-overview-grid-item")][strong[. = "' + label + '"]]'
    elements = driver.find_elements('xpath', xpath)
    if len(elements) == 0:
        return ""
    value = elements[0].text.replace(label, "", 1)
    value = " ".join(value.split())
    return value

# Extract the summary paragraph of the policy page
def get_summary():
    elements = driver.find_elements('xpath', '//div[@class = "policy-summary"]/p')
    if len(elements) == 0:
        return ""
    return elements[0].text

time.sleep(10)
driver.get(list_urls[0])

print(get_field("Jurisdiction Name"))
print(get_field("Jurisdiction Type"))
print(get_field("State"))
print(get_field("Food System Category"))
print(get_summary())

Abingdon
Town/Township
Virginia
Distribute; Get
This provision exempts people selling fresh farm products (among other exemptions) from a requirement that mobile peddlers, vendors, and canvassers obtain a permit to operate within the town.


# <span style="color:darkblue"> V. Loop Over All Policies </span>

<font size = "5">

Visit each policy page and collect the same information

<font size = "3">

- For each saved web address, I open the policy page and collect the same fields.
- I put each policy's information into one record and add it to a list, which holds all the records until they are turned into a table.
- I wait 10 seconds before each policy-page request, as requested by the site's `robots.txt` (the site's instructions for automated visitors).
- This simple version does not retry a page if an error occurs. The error will appear, and the collection may stop.
- To test a few pages first, change `num_to_scrape` to a small number such as 5.

Each record uses the same field names, so the information will line up in the final table. The ten-second pause follows the site's request and spaces out the page visits.

I kept this version simple instead of adding automatic retries or backup files. If a page fails, the error appears and may stop the run. The progress message shows how many pages the loop has reached, but it does not confirm that every policy was collected.

The message `Collected 1 of 605` means the loop has reached its first policy out of 605 planned policies. Python starts counting positions at 0, so `i + 1` displays a person-friendly count starting at 1. `num_to_scrape` supplies the total number planned for this run. I print this after each policy is added to the list so I can see the run's progress while it works.

In [21]:
num_to_scrape = num_links

data = []

for i in range(num_to_scrape):
    url = list_urls[i]

    time.sleep(10)
    driver.get(url)

    data.append({"jurisdiction": get_field("Jurisdiction Name"),
                 "jurisdiction_type": get_field("Jurisdiction Type"),
                 "state": get_field("State"),
                 "policy_category": get_field("Food System Category"),
                 "summary": get_summary(),
                 "url": url})

    print("Collected", i + 1, "of", num_to_scrape)

Collected 1 of 605
Collected 2 of 605
Collected 3 of 605
Collected 4 of 605
Collected 5 of 605
Collected 6 of 605
Collected 7 of 605
Collected 8 of 605
Collected 9 of 605
Collected 10 of 605
Collected 11 of 605
Collected 12 of 605
Collected 13 of 605
Collected 14 of 605
Collected 15 of 605
Collected 16 of 605
Collected 17 of 605
Collected 18 of 605
Collected 19 of 605
Collected 20 of 605
Collected 21 of 605
Collected 22 of 605
Collected 23 of 605
Collected 24 of 605
Collected 25 of 605
Collected 26 of 605
Collected 27 of 605
Collected 28 of 605
Collected 29 of 605
Collected 30 of 605
Collected 31 of 605
Collected 32 of 605
Collected 33 of 605
Collected 34 of 605
Collected 35 of 605
Collected 36 of 605
Collected 37 of 605
Collected 38 of 605
Collected 39 of 605
Collected 40 of 605
Collected 41 of 605
Collected 42 of 605
Collected 43 of 605
Collected 44 of 605
Collected 45 of 605
Collected 46 of 605
Collected 47 of 605
Collected 48 of 605
Collected 49 of 605
Collected 50 of 605
Collected

InvalidSessionIdException: Message: invalid session id: session deleted as the browser has closed the connection
from disconnected: Unable to receive message from renderer
  (Session info: chrome=155.0.8059.85); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#invalidsessionidexception
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	(No symbol) [0x7ff79684e745]
	(No symbol) [0x7ff796403d00]
	(No symbol) [0x7ff796233f3d]
	(No symbol) [0x7ff796221329]
	(No symbol) [0x7ff796221011]
	(No symbol) [0x7ff79621ecfa]
	(No symbol) [0x7ff79621f452]
	(No symbol) [0x7ff796238237]
	(No symbol) [0x7ff7962e5914]
	(No symbol) [0x7ff7962b8ada]
	(No symbol) [0x7ff7962e4b5a]
	(No symbol) [0x7ff796281462]
	(No symbol) [0x7ff796282273]
	(No symbol) [0x7ff7968f17b1]
	(No symbol) [0x7ff7969287c7]
	(No symbol) [0x7ff79691c85e]
	(No symbol) [0x7ff7965c884e]
	(No symbol) [0x7ff796410415]
	(No symbol) [0x7ff79640bdf4]
	(No sym

# <span style="color:darkblue"> VI. Convert to Pandas DataFrame and Export </span>

I turn the list of records into a pandas DataFrame, which is a table with rows and columns. This makes the results easier to review and use in later analysis. I save the table as a CSV file so it can be opened outside this notebook. Finally, `driver.quit()` closes Chrome.

In [ ]:
data_df = pd.DataFrame(data)

# Close the browser
driver.quit()

# Save to a CSV file
data_df.to_csv("policy_data_selenium.csv", index = False)

# Report the number of observations
print("Number of policies collected:", len(data_df))

# Report fields that the website leaves blank
print(data_df.eq("").sum())

data_df.head()

## Data collection notes

### Interpreting the output

The CSV contains only the fields I chose to collect, not everything shown on the website. The blank-value counts show how often no text was collected for each field. A blank does not tell us why the information was missing from the page.

### Limitations and reproducibility

- The page-finding directions are based on how the website was organized when I wrote this notebook. If the website changes, I may need to update them.
- The notebook waits 10 seconds between policy-page requests, but it does not check whether a page has fully loaded. A slow page or an error may interrupt collection.
- To try the process, first collect a few pages by setting `num_to_scrape` to 5. For all the links, set it to `num_links`. Then check the number of rows and the blank-value counts.
- The website may be updated, so the results can differ between collection dates. Keep the CSV and note when it was collected if you want to compare runs.


- Source: Healthy Food Policy Project policy database (permission obtained). The policy fields replace the product fields in the general project example.
- Restrictions: ``` robots.txt ``` allows policy pages and asks for a 10-second crawl delay, which the code follows.
- Blank values: some policies have no category, jurisdiction type, or summary on the website. These stay blank rather than being filled in.
- Output: ``` policy_data_selenium.csv ```.